# Train/test leakage analysis (StratifiedGroupKFold)

Same as `leakage_analysis_minimal.ipynb`, except the grouped splits use
`StratifiedGroupKFold` instead of `GroupShuffleSplit` -- it keeps whole
lecture/course groups on one side like GSS does, but also tries to balance
`lecturer_id` across train/test the way `stratify=` does for the random split.

It's fold-based rather than a direct `test_size`, so `n_splits=5` stands in
for an ~80/20 split. It's a best-effort balance, not a guarantee: if a
lecturer's sentences are concentrated in only one or two courses, no split
can keep them on both sides -- that's a structural confound in the data, not
something a smarter splitter can fix. Expect `grouped_by_course` to still show
lecturers missing from test; the main effect should show up in
`grouped_by_lecture`.

In [5]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score

data = pd.read_csv('../sentences_top100.csv').dropna(subset=['content'])
data = data[data['content'].str.strip() != ''].reset_index(drop=True)
data.shape

(4317765, 6)

In [6]:
ALL_IDS = sorted(data['lecturer_id'].unique())  # fixed label set so macro_f1 is comparable across splits


def run_split(name, group_col):
    # group_col=None -> random 80/20 split over individual sentences (the leaky baseline).
    # else -> StratifiedGroupKFold on that column: whole lecture/course stays on one
    # side (like GroupShuffleSplit), while also trying to balance 80/20 across lecturers
    # n_splits=5 stands in for ~80/20
    # since this splitter is fold-based, not test_size-based.
    if group_col is None:
        train_idx, test_idx = train_test_split(
            np.arange(len(data)), test_size=0.2, random_state=42, stratify=data['lecturer_id'],
        )
    else:
        sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
        train_idx, test_idx = next(sgkf.split(data, y=data['lecturer_id'], groups=data[group_col]))

    X_train, X_test = data['content'].iloc[train_idx], data['content'].iloc[test_idx]
    y_train, y_test = data['lecturer_id'].iloc[train_idx], data['lecturer_id'].iloc[test_idx]

    vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=5, max_df=0.5, max_features=50_000)
    clf = MultinomialNB().fit(vectorizer.fit_transform(X_train), y_train)
    y_pred = clf.predict(vectorizer.transform(X_test))

    return {
        'split': name,
        'train_size': len(train_idx),
        'test_size': len(test_idx),
        'baseline_accuracy': y_test.value_counts(normalize=True).iloc[0],  # always-predict-majority-class
        'accuracy': accuracy_score(y_test, y_pred),
        'macro_f1': f1_score(y_test, y_pred, labels=ALL_IDS, average='macro', zero_division=0),
        'lecturers_absent_from_test': len(set(ALL_IDS) - set(y_test.unique())),  # zero-support classes
    }

In [7]:
splits = [
    ('random_by_sentence', None),
    ('grouped_by_lecture', 'lecture_id'),
    ('grouped_by_course', 'course_id'),
]

comparison = pd.DataFrame([run_split(name, col) for name, col in splits])
comparison.to_csv('topic_leakage.csv', index=False)
comparison

,split,train_size,test_size,baseline_accuracy,accuracy,macro_f1,lecturers_absent_from_test
0,random_by_sentence,3454212,863553,0.176267,0.446547,0.112103,0
1,grouped_by_lecture,3453202,864563,0.175838,0.432495,0.102008,1
2,grouped_by_course,3458300,859465,0.174631,0.416317,0.073090,25
